In [ ]:
import pandas as pd
import numpy as np
import gc
import os
import json
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from google.colab import drive

# 1. SETUP
drive.mount('/content/drive', force_remount=True)
base_path = "/content/drive/MyDrive/MLA_Final_Project/Datasets_Final/"

# 2. LOAD DATA
print("📂 Loading sanitized dataset...")
df = pd.read_parquet(os.path.join(base_path, "full_data.parquet"))

# 3. FEATURE SELECTION (ANOVA F-TEST)
X = df.drop(columns=['Label'])
y = df['Label']

print("⚡ Running ANOVA F-Test to select top 25 features...")
selector = SelectKBest(score_func=f_classif, k=25)
selector.fit(X, y)
selected_features = X.columns[selector.get_support()].tolist()

# Save feature list for the App
with open(os.path.join(base_path, "selected_features.json"), "w") as f:
    json.dump(selected_features, f)

# 4. TRAIN-TEST SPLIT (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X[selected_features], y, test_size=0.2, random_state=42, stratify=y
)

# 5. CLASS BALANCING (SMOTE)
print(f"\n🔄 Applying SMOTE Balancing...")
# Capturing counts before SMOTE
before_normal = sum(y_train == 0)
before_attack = sum(y_train == 1)

start_smote = time.time()
# sampling_strategy='auto' or 1.0 makes classes exactly equal
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
smote_time = (time.time() - start_smote) / 60

# Capturing counts after SMOTE
after_normal = sum(y_train_resampled == 0)
after_attack = sum(y_train_resampled == 1)

# 6. SAVE FOR PHASE 3
np.save(os.path.join(base_path, "X_train.npy"), X_train_resampled)
np.save(os.path.join(base_path, "y_train.npy"), y_train_resampled)
np.save(os.path.join(base_path, "X_test.npy"), X_test.values)
np.save(os.path.join(base_path, "y_test.npy"), y_test.values)

# --- 🚀 PROFESSOR-LEVEL INSIGHT: THE BALANCING REPORT ---
print("\n" + "="*60)
print("📊 PHASE 2: CLASS BALANCING & FEATURE AUDIT")
print("="*60)
print(f"{'CLASS TYPE':<20} | {'BEFORE SMOTE':<15} | {'AFTER SMOTE':<15}")
print("-" * 60)
print(f"{'Normal (Benign)':<20} | {before_normal:<15,} | {after_normal:<15,}")
print(f"{'Attack (Anomaly)':<20} | {before_attack:<15,} | {after_attack:<15,}")
print("-" * 60)
print(f"✅ Status: Dataset is now 100% Balanced.")
print(f"⏱️ SMOTE Runtime: {smote_time:.2f} minutes")
print(f"🎯 Features Retained: 25 (Optimized via ANOVA)")
print("\n📝 JUSTIFICATION: SMOTE has synthetically augmented the 'Attack'")
print("class using K-Nearest Neighbors. This ensures the CNN-LSTM")
print("prioritizes malicious patterns equally with normal traffic.")
print("="*60)

# Final Cleanup
del df, X, y, X_train, X_train_resampled
gc.collect()

Mounted at /content/drive
📂 Loading sanitized dataset...
⚡ Running ANOVA F-Test to select top 25 features...

🔄 Applying SMOTE Balancing...

📊 PHASE 2: CLASS BALANCING & FEATURE AUDIT
CLASS TYPE           | BEFORE SMOTE    | AFTER SMOTE    
------------------------------------------------------------
Normal (Benign)      | 1,465,234       | 1,465,234      
Attack (Anomaly)     | 243,866         | 1,465,234      
------------------------------------------------------------
✅ Status: Dataset is now 100% Balanced.
⏱️ SMOTE Runtime: 4.59 minutes
🎯 Features Retained: 25 (Optimized via ANOVA)

📝 JUSTIFICATION: SMOTE has synthetically augmented the 'Attack'
class using K-Nearest Neighbors. This ensures the CNN-LSTM
prioritizes malicious patterns equally with normal traffic.


0